# 11 · Feature consistency check
Charge un run JSON et compare les features offline vs celles générées par la pipeline online (simulation).

In [1]:
import json
from pathlib import Path
import pandas as pd

from src.analysis.feature_consistency import (
    matches_from_artifact,
    collect_offline_features,
    collect_online_features,
    compare_feature_frames,
)

/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RUN_PATH = Path('data/mass_prediction_nba/runs/2025-11-23/nba-run-20251123-132543-307075.json')
TOLERANCE = 1e-6

artifact = json.loads(RUN_PATH.read_text())
match_keys = matches_from_artifact(artifact)
print(f'Matches analysés: {len(match_keys)}')

Matches analysés: 1


In [3]:
offline_df = collect_offline_features(match_keys)
online_df = collect_online_features(match_keys)
print('Offline shape:', offline_df.shape)
print('Online shape:', online_df.shape)

/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "vector_enabled" in "SqliteOnlineStoreConfig" shadows an attribute in parent "VectorStoreConfig"
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/feast/repo_config.py:268: DeprecationWarning: The serialization version 2 and below will be deprecated in the next release. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/.venv/lib/python3.12/site-packages/feast/repo_config.py:268: DeprecationWarning: The serialization version 2 and below will be deprecated in the next release. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(
/home/ju/Documents/Dev/NBA_Predictor/src/prediction/point_total.py:272: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages

Offline shape: (1, 4143)
Online shape: (1, 4143)


In [4]:
diff_df = compare_feature_frames(offline_df, online_df, tolerance=TOLERANCE)
diff_df.head()

,match_key,max_abs_diff,num_columns_over_tol
0,"(1610612755, 1610612748, 2025-11-23)",0.000038,371


In [5]:
if not diff_df.empty:
    key = diff_df.iloc[0]['match_key']
    offline_row = offline_df.loc[key]
    online_row = online_df.loc[key]
    comparison = pd.DataFrame({
        'offline': offline_row,
        'online': online_row,
        'abs_diff': (offline_row - online_row).abs(),
    })
    display(comparison.sort_values('abs_diff', ascending=False).head(20))
else:
    print('Aucune différence détectée (<= tolérance).')

,offline,online,abs_diff
AWAY_OPP_ELO_PRE,1396.988563,1396.988525,0.000038
HOME_ELO_PRE,1396.988563,1396.988525,0.000038
HOME_OPP_ELO_PRE,1526.444247,1526.444214,0.000033
AWAY_ELO_PRE,1526.444247,1526.444214,0.000033
MATCH_ELO_LEVEL_AVG,1461.716405,1461.716431,0.000026
AWAY_ELO_PRE_SEASON,1556.129776,1556.129761,0.000015
HOME_OPP_ELO_PRE_SEASON,1556.129776,1556.129761,0.000015
HOME_ROLL_MINUTES_PLAYED_200,449.669542,449.669556,0.000014
HOME_H2H_LAST_50_AVG_TOTAL_POINTS,423.920000,423.920013,0.000013
AWAY_H2H_LAST_50_AVG_TOTAL_POINTS,423.920000,423.920013,0.000013
